# Redes Neuronales Recurrentes (RNN)

## ¿Cuál es el objetivo de una RNN?

Las **Redes Neuronales Recurrentes** (Recurrent Neural Networks, RNN) son un tipo de red neuronal diseñada para procesar **datos secuenciales**, donde el **orden** de los elementos importa.

A diferencia de una red feedforward (MLP o CNN) que procesa cada entrada de forma independiente, una RNN tiene **memoria**: al procesar cada elemento de la secuencia, recuerda lo que vio antes.

**¿Por qué necesitamos memoria?**

Muchos datos del mundo real son secuencias donde el contexto pasado importa:

| Tipo de dato | Ejemplo | ¿Por qué importa el orden? |
|---|---|---|
| Texto | "El gato se sentó en la ____" | La palabra siguiente depende de las anteriores |
| Series temporales | Precio de acciones día a día | El precio de mañana depende del patrón histórico |
| Audio | Señal de voz | Los fonemas dependen de los sonidos previos |
| ADN | Secuencia de nucleótidos | La función depende del orden de las bases |

Formalmente, dado una secuencia de entrada $\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_T$, la RNN mantiene un **estado oculto** $\mathbf{h}_t$ que se actualiza en cada paso temporal:

$$
\mathbf{h}_t = f(\mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{W}_{xh} \mathbf{x}_t + \mathbf{b}_h)
$$

$$
\hat{\mathbf{y}}_t = g(\mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y)
$$

El estado oculto $\mathbf{h}_t$ actúa como la **memoria** de la red: resume toda la información relevante de la secuencia hasta el paso $t$.

**Ejemplo simple: predecir la siguiente letra**

Consideremos el problema de predecir la siguiente letra en una secuencia. Dado el vocabulario `{h, e, l, o}` y la palabra `"hello"`:

| Paso $t$ | Entrada $x_t$ | Contexto (lo que la red "recuerda") | Salida esperada $y_t$ |
|---|---|---|---|
| 1 | `h` | — | `e` |
| 2 | `e` | `h` | `l` |
| 3 | `l` | `he` | `l` |
| 4 | `l` | `hel` | `o` |

La RNN recibe una letra a la vez y debe predecir la siguiente. Para hacerlo correctamente, **necesita recordar** las letras anteriores.

Nótese que en el paso 3 y 4 la entrada es la misma (`l`), pero la salida esperada es diferente (`l` vs `o`). Esto es imposible sin memoria — la red necesita saber qué vino antes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualizar el concepto de RNN vs Feedforward
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Feedforward (sin memoria) ---
ax = axes[0]
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Red Feedforward\n(Sin memoria — cada entrada es independiente)', fontsize=11, fontweight='bold')

for i, (letter, y_out) in enumerate(zip(['h','e','l','l'], ['e','l','l','o'])):
    x_pos = i * 2.8
    # Entrada
    ax.text(x_pos+0.5, 0, f'"{letter}"', ha='center', fontsize=14, fontweight='bold', color='#3498db')
    # Caja de red
    rect = mpatches.FancyBboxPatch((x_pos-0.1, 1.2), 1.2, 1, boxstyle='round,pad=0.1',
                                    facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_pos+0.5, 1.7, 'NN', ha='center', fontsize=11, fontweight='bold')
    # Flecha entrada → red
    ax.annotate('', xy=(x_pos+0.5, 1.2), xytext=(x_pos+0.5, 0.4),
                arrowprops=dict(arrowstyle='->', color='#3498db', lw=1.5))
    # Salida
    ax.annotate('', xy=(x_pos+0.5, 3.5), xytext=(x_pos+0.5, 2.3),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))
    ax.text(x_pos+0.5, 3.8, f'"{y_out}"', ha='center', fontsize=14, fontweight='bold', color='#e74c3c')

ax.text(5, 4.5, '✗ No puede distinguir "hel" + l → l  de  "hell" + l → o',
        ha='center', fontsize=9, color='red', fontstyle='italic')

# --- RNN (con memoria) ---
ax = axes[1]
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Red Recurrente (RNN)\n(Con memoria — recuerda el contexto)', fontsize=11, fontweight='bold')

for i, (letter, y_out, mem) in enumerate(zip(
    ['h','e','l','l'],
    ['e','l','l','o'],
    ['','h','he','hel'])):
    x_pos = i * 2.8
    # Entrada
    ax.text(x_pos+0.5, 0, f'"{letter}"', ha='center', fontsize=14, fontweight='bold', color='#3498db')
    # Caja de red
    rect = mpatches.FancyBboxPatch((x_pos-0.1, 1.2), 1.2, 1, boxstyle='round,pad=0.1',
                                    facecolor='#d5f5e3', edgecolor='#27ae60', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_pos+0.5, 1.7, f'$h_{i+1}$', ha='center', fontsize=12, fontweight='bold')
    # Flecha entrada → red
    ax.annotate('', xy=(x_pos+0.5, 1.2), xytext=(x_pos+0.5, 0.4),
                arrowprops=dict(arrowstyle='->', color='#3498db', lw=1.5))
    # Salida
    ax.annotate('', xy=(x_pos+0.5, 3.5), xytext=(x_pos+0.5, 2.3),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))
    ax.text(x_pos+0.5, 3.8, f'"{y_out}"', ha='center', fontsize=14, fontweight='bold', color='#e74c3c')
    # Conexión recurrente
    if i < 3:
        ax.annotate('', xy=(x_pos+2.8-0.1, 1.7), xytext=(x_pos+1.1, 1.7),
                    arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))
        ax.text(x_pos+1.6, 1.95, f'$h_{i+1}$', fontsize=8, color='#27ae60')

ax.text(5, 4.5, '✓ Usa el estado oculto h para recordar el contexto anterior',
        ha='center', fontsize=9, color='green', fontstyle='italic')

plt.tight_layout()
plt.show()

## ¿Cómo procesa una RNN las secuencias?

La RNN procesa la secuencia **un elemento a la vez**, manteniendo un estado oculto que se pasa de un paso al siguiente como una "memoria". Pero... **¿cuáles son las ecuaciones exactas? ¿Cómo se entrena?** Necesitamos la demostración matemática completa.

## Demostración Matemática

### Paso 1: Las ecuaciones de la RNN

Una RNN tiene tres conjuntos de pesos que comparte a lo largo de toda la secuencia:

- $\mathbf{W}_{xh}$: pesos de entrada → estado oculto (dimensión: $d_h \times d_x$)
- $\mathbf{W}_{hh}$: pesos de estado oculto → estado oculto (dimensión: $d_h \times d_h$)
- $\mathbf{W}_{hy}$: pesos de estado oculto → salida (dimensión: $d_y \times d_h$)

En cada paso temporal $t$:

**1. Actualizar el estado oculto:**

$$
\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{b}_h)
$$

**2. Calcular la salida:**

$$
\mathbf{o}_t = \mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y
$$

**3. Aplicar softmax** (para clasificación):

$$
\hat{\mathbf{y}}_t = \text{softmax}(\mathbf{o}_t)
$$

Donde:
- $\mathbf{x}_t \in \mathbb{R}^{d_x}$: entrada en el paso $t$ (ej. one-hot de un carácter)
- $\mathbf{h}_t \in \mathbb{R}^{d_h}$: estado oculto en el paso $t$ (la "memoria")
- $\mathbf{h}_0 = \mathbf{0}$: estado inicial (ceros)
- $\hat{\mathbf{y}}_t \in \mathbb{R}^{d_y}$: probabilidades de salida
- $\tanh$: función de activación que mantiene los valores en $(-1, 1)$

**¿Por qué tanh y no ReLU?**

La tangente hiperbólica está acotada en $(-1, 1)$, lo que ayuda a evitar que los valores del estado oculto crezcan sin control al recircular por la red. ReLU no tiene esta propiedad, aunque se usa en variantes más modernas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizar cómo fluye la información en una RNN
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('RNN "desenrollada" (Unrolled) a lo largo del tiempo', fontsize=14, fontweight='bold')

steps = ['$t=1$', '$t=2$', '$t=3$', '$...$', '$t=T$']
x_labels = ['$\mathbf{x}_1$', '$\mathbf{x}_2$', '$\mathbf{x}_3$', '$...$', '$\mathbf{x}_T$']
h_labels = ['$\mathbf{h}_1$', '$\mathbf{h}_2$', '$\mathbf{h}_3$', '$...$', '$\mathbf{h}_T$']
y_labels = ['$\hat{\mathbf{y}}_1$', '$\hat{\mathbf{y}}_2$', '$\hat{\mathbf{y}}_3$', '$...$', '$\hat{\mathbf{y}}_T$']

for i, (step, xl, hl, yl) in enumerate(zip(steps, x_labels, h_labels, y_labels)):
    x_pos = i * 3.2 + 0.5
    
    if i == 3:  # ...
        ax.text(x_pos, 3, '...', fontsize=24, ha='center', va='center', color='gray')
        continue
    
    # Entrada x_t
    ax.text(x_pos, 0.3, xl, ha='center', fontsize=13, color='#3498db', fontweight='bold')
    
    # Flecha x → h
    ax.annotate('', xy=(x_pos, 2.2), xytext=(x_pos, 0.8),
                arrowprops=dict(arrowstyle='->', color='#3498db', lw=2))
    ax.text(x_pos - 0.7, 1.5, '$W_{xh}$', fontsize=9, color='#8e44ad')
    
    # Nodo h_t
    circle = plt.Circle((x_pos, 3), 0.7, color='#27ae60', alpha=0.2, ec='#27ae60', lw=2)
    ax.add_patch(circle)
    ax.text(x_pos, 3, hl, ha='center', va='center', fontsize=12, fontweight='bold')
    
    # Flecha h → y
    ax.annotate('', xy=(x_pos, 5.0), xytext=(x_pos, 3.8),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
    ax.text(x_pos - 0.7, 4.3, '$W_{hy}$', fontsize=9, color='#8e44ad')
    
    # Salida y_t
    ax.text(x_pos, 5.5, yl, ha='center', fontsize=13, color='#e74c3c', fontweight='bold')
    
    # Paso temporal
    ax.text(x_pos, -0.5, step, ha='center', fontsize=10, color='gray')

# Flechas recurrentes h_t-1 → h_t
# h0 → h1
ax.text(-0.5, 3, '$\mathbf{h}_0$', ha='center', fontsize=11, color='gray')
ax.annotate('', xy=(0.5-0.7, 3), xytext=(-0.1, 3),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))

for i in range(3):
    if i < 2:
        x1 = i * 3.2 + 0.5 + 0.7
        x2 = (i+1) * 3.2 + 0.5 - 0.7
    else:
        x1 = i * 3.2 + 0.5 + 0.7
        x2 = (i+1) * 3.2 + 0.5
    ax.annotate('', xy=(x2, 3), xytext=(x1, 3),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))
    ax.text((x1+x2)/2, 3.5, '$W_{hh}$', fontsize=9, color='#8e44ad', ha='center')

# ... → h_T
ax.annotate('', xy=(4*3.2+0.5-0.7, 3), xytext=(3*3.2+0.5+0.5, 3),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))

plt.tight_layout()
plt.show()

### Paso 2: Función de Pérdida

La pérdida total de la secuencia es la suma de las pérdidas en cada paso temporal:

$$
\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t = -\sum_{t=1}^{T} \sum_{c=1}^{C} y_{t,c} \log(\hat{y}_{t,c})
$$

Para el caso de predicción del siguiente carácter, donde $y_t$ es un vector one-hot del carácter correcto:

$$
\mathcal{L}_t = -\log(\hat{y}_{t, \text{correcto}})
$$

Es decir, queremos que la probabilidad asignada al carácter correcto sea lo más alta posible en cada paso.

### Paso 3: Backpropagation Through Time (BPTT)

Para entrenar la RNN, necesitamos calcular los gradientes de $\mathcal{L}$ respecto a todos los pesos. El algoritmo se llama **Backpropagation Through Time** (BPTT) porque propagamos el error hacia atrás **a través del tiempo**.

#### Gradiente de $\mathbf{W}_{hy}$ (salida):

Este es el más simple, ya que solo depende del paso actual:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{hy}} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial \mathbf{W}_{hy}} = \sum_{t=1}^{T} (\hat{\mathbf{y}}_t - \mathbf{y}_t) \mathbf{h}_t^T
$$

#### Gradiente de $\mathbf{W}_{hh}$ y $\mathbf{W}_{xh}$ (recurrencia):

Aquí es donde se complica. Como $\mathbf{h}_t$ depende de $\mathbf{h}_{t-1}$, que depende de $\mathbf{h}_{t-2}$, etc., necesitamos la regla de la cadena a lo largo del tiempo:

$$
\frac{\partial \mathcal{L}_t}{\partial \mathbf{W}_{hh}} = \sum_{k=1}^{t} \frac{\partial \mathcal{L}_t}{\partial \hat{\mathbf{y}}_t} \frac{\partial \hat{\mathbf{y}}_t}{\partial \mathbf{h}_t} \left(\prod_{j=k+1}^{t} \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}\right) \frac{\partial \mathbf{h}_k}{\partial \mathbf{W}_{hh}}
$$

Donde la derivada del estado oculto es:

$$
\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \text{diag}(1 - \mathbf{h}_t^2) \cdot \mathbf{W}_{hh}
$$

(el $1 - \mathbf{h}_t^2$ viene de la derivada de $\tanh$)

En la práctica, el gradiente se propaga hacia atrás paso a paso:

$$
\delta_t = \frac{\partial \mathcal{L}_t}{\partial \mathbf{h}_t}, \quad \delta_{t-1} = \mathbf{W}_{hh}^T \delta_t \odot (1 - \mathbf{h}_{t-1}^2)
$$

### Paso 4: El Problema del Gradiente Desvaneciente

Observemos el producto de la cadena:

$$
\prod_{j=k+1}^{t} \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}} = \prod_{j=k+1}^{t} \text{diag}(1 - \mathbf{h}_j^2) \cdot \mathbf{W}_{hh}
$$

Este es un **producto de muchas matrices**. Si los valores propios de $\mathbf{W}_{hh}$ son:

- **Menores que 1**: el producto tiende a $\mathbf{0}$ exponencialmente → **gradiente desvaneciente** (vanishing gradient)
- **Mayores que 1**: el producto tiende a $\infty$ → **gradiente explosivo** (exploding gradient)

Más formalmente, si $\lambda_{\max}$ es el valor propio más grande de $\mathbf{W}_{hh}$:

$$
\left\|\prod_{j=k+1}^{t} \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}\right\| \approx \lambda_{\max}^{t-k}
$$

**Consecuencias:**
- Con **gradiente desvaneciente**: la RNN no puede aprender dependencias a largo plazo ("olvida" el pasado lejano)
- Con **gradiente explosivo**: el entrenamiento diverge (los pesos se van a infinito)

**Soluciones:**

| Problema | Solución |
|---|---|
| Gradiente explosivo | **Gradient clipping**: recortar el gradiente si su norma excede un umbral |
| Gradiente desvaneciente | **LSTM/GRU**: arquitecturas con compuertas que controlan el flujo de información |

Implementaremos **gradient clipping** en nuestra RNN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizar el problema del gradiente desvaneciente
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simular producto de matrices con diferentes valores propios
steps = np.arange(1, 30)

# Gradiente desvaneciente (λ < 1)
for lam in [0.9, 0.7, 0.5]:
    grad_magnitude = lam ** steps
    axes[0].plot(steps, grad_magnitude, 'o-', label=f'λ = {lam}', markersize=3)

axes[0].set_xlabel('Pasos hacia atrás (t - k)', fontsize=11)
axes[0].set_ylabel('Magnitud del gradiente $λ^{t-k}$', fontsize=11)
axes[0].set_title('Gradiente Desvaneciente (λ < 1)', fontsize=12, fontweight='bold', color='#e74c3c')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')
axes[0].axhline(y=1e-5, color='red', ls='--', alpha=0.5)
axes[0].text(15, 2e-5, 'Gradiente ≈ 0\n(no aprende)', color='red', fontsize=9)

# Gradiente explosivo (λ > 1)
steps_short = np.arange(1, 15)
for lam in [1.1, 1.3, 1.5]:
    grad_magnitude = lam ** steps_short
    axes[1].plot(steps_short, grad_magnitude, 'o-', label=f'λ = {lam}', markersize=3)

axes[1].set_xlabel('Pasos hacia atrás (t - k)', fontsize=11)
axes[1].set_ylabel('Magnitud del gradiente $λ^{t-k}$', fontsize=11)
axes[1].set_title('Gradiente Explosivo (λ > 1)', fontsize=12, fontweight='bold', color='#e67e22')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=100, color='orange', ls='--', alpha=0.5)
axes[1].text(7, 120, 'Gradiente → ∞\n(entrenamiento diverge)', color='orange', fontsize=9)

plt.suptitle('El Problema Fundamental de las RNNs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Paso 5: Algoritmo Completo (BPTT)

El algoritmo completo de entrenamiento paso a paso:

---

**Entrada:** Secuencia $(\mathbf{x}_1, \dots, \mathbf{x}_T)$, etiquetas $(\mathbf{y}_1, \dots, \mathbf{y}_T)$, tasa $\eta$

**Forward Pass:**

$\quad$ $\mathbf{h}_0 = \mathbf{0}$

$\quad$ **Para** $t = 1, \dots, T$:

$\quad\quad$ $\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{b}_h)$

$\quad\quad$ $\hat{\mathbf{y}}_t = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y)$

**Backward Pass (BPTT):**

$\quad$ Inicializar: $d\mathbf{W}_{xh} = \mathbf{0}$, $d\mathbf{W}_{hh} = \mathbf{0}$, $d\mathbf{W}_{hy} = \mathbf{0}$, $d\mathbf{h}_{\text{next}} = \mathbf{0}$

$\quad$ **Para** $t = T, T-1, \dots, 1$:

$\quad\quad$ $d\mathbf{o}_t = \hat{\mathbf{y}}_t - \mathbf{y}_t$

$\quad\quad$ $d\mathbf{W}_{hy} \mathrel{+}= d\mathbf{o}_t \cdot \mathbf{h}_t^T$

$\quad\quad$ $d\mathbf{h}_t = \mathbf{W}_{hy}^T d\mathbf{o}_t + d\mathbf{h}_{\text{next}}$

$\quad\quad$ $d\mathbf{h}_{\text{raw}} = (1 - \mathbf{h}_t^2) \odot d\mathbf{h}_t$

$\quad\quad$ $d\mathbf{W}_{xh} \mathrel{+}= d\mathbf{h}_{\text{raw}} \cdot \mathbf{x}_t^T$

$\quad\quad$ $d\mathbf{W}_{hh} \mathrel{+}= d\mathbf{h}_{\text{raw}} \cdot \mathbf{h}_{t-1}^T$

$\quad\quad$ $d\mathbf{h}_{\text{next}} = \mathbf{W}_{hh}^T d\mathbf{h}_{\text{raw}}$

**Gradient Clipping:** Si $\|d\mathbf{W}\| > \text{clip}$: $d\mathbf{W} \leftarrow \text{clip} \cdot \frac{d\mathbf{W}}{\|d\mathbf{W}\|}$

**Actualizar pesos:** $\mathbf{W} \leftarrow \mathbf{W} - \eta \cdot d\mathbf{W}$ para todos los pesos

---

### Ejemplo numérico

Veamos un forward pass con datos concretos. Usaremos el vocabulario `{h, e, l, o}` con la secuencia `"hell"` → `"ello"`.

**Codificación one-hot:**

$$
h = \begin{bmatrix}1\\0\\0\\0\end{bmatrix}, \quad
e = \begin{bmatrix}0\\1\\0\\0\end{bmatrix}, \quad
l = \begin{bmatrix}0\\0\\1\\0\end{bmatrix}, \quad
o = \begin{bmatrix}0\\0\\0\\1\end{bmatrix}
$$

**Dimensiones:** $d_x = 4$ (vocabulario), $d_h = 3$ (estado oculto), $d_y = 4$ (vocabulario)

Supongamos pesos iniciales aleatorios (simplificados):

$$
\mathbf{W}_{xh} = \begin{bmatrix} 0.1 & 0.2 & -0.1 & 0.3 \\ -0.2 & 0.1 & 0.3 & -0.1 \\ 0.3 & -0.1 & 0.2 & 0.1 \end{bmatrix}, \quad
\mathbf{W}_{hh} = \begin{bmatrix} 0.1 & 0.2 & -0.1 \\ -0.1 & 0.1 & 0.2 \\ 0.2 & -0.1 & 0.1 \end{bmatrix}
$$

In [ ]:
import numpy as np

# Ejemplo numérico paso a paso
np.set_printoptions(precision=4, suppress=True)

# Vocabulario
vocab = {'h': 0, 'e': 1, 'l': 2, 'o': 3}
idx_to_char = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)
hidden_size = 3

# One-hot encoding
def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1.0
    return v

# Pesos (fijos para el ejemplo)
np.random.seed(42)
W_xh = np.random.randn(hidden_size, vocab_size) * 0.3
W_hh = np.random.randn(hidden_size, hidden_size) * 0.3
W_hy = np.random.randn(vocab_size, hidden_size) * 0.3
b_h = np.zeros(hidden_size)
b_y = np.zeros(vocab_size)

def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

# Secuencia: "hell" → "ello"
input_seq = [vocab['h'], vocab['e'], vocab['l'], vocab['l']]
target_seq = [vocab['e'], vocab['l'], vocab['l'], vocab['o']]

print("="*60)
print("FORWARD PASS DE LA RNN")
print("="*60)
print(f"Secuencia de entrada: {''.join([idx_to_char[i] for i in input_seq])}")
print(f"Secuencia objetivo:   {''.join([idx_to_char[i] for i in target_seq])}")
print(f"\nDimensiones: vocab={vocab_size}, hidden={hidden_size}")
print()

h = np.zeros(hidden_size)  # h_0
total_loss = 0

for t, (x_idx, y_idx) in enumerate(zip(input_seq, target_seq)):
    x_t = one_hot(x_idx, vocab_size)
    
    # Paso 1: Estado oculto
    z_h = W_xh @ x_t + W_hh @ h + b_h
    h_new = np.tanh(z_h)
    
    # Paso 2: Salida
    o_t = W_hy @ h_new + b_y
    probs = softmax(o_t)
    
    # Pérdida
    loss_t = -np.log(probs[y_idx] + 1e-10)
    total_loss += loss_t
    
    print(f"--- Paso t={t+1}: entrada='{idx_to_char[x_idx]}' ---")
    print(f"  x_t = {x_t} (one-hot de '{idx_to_char[x_idx]}')")
    print(f"  h_{t} (anterior) = {h}")
    print(f"  z_h = W_xh·x + W_hh·h + b = {z_h}")
    print(f"  h_{t+1} = tanh(z_h) = {h_new}")
    print(f"  Probabilidades: {dict(zip(vocab.keys(), [f'{p:.4f}' for p in probs]))}")
    print(f"  Predicción: '{idx_to_char[np.argmax(probs)]}' (prob={probs[np.argmax(probs)]:.4f})")
    print(f"  Objetivo: '{idx_to_char[y_idx]}' → Loss_t = {loss_t:.4f}")
    print()
    
    h = h_new

print(f"Loss total = {total_loss:.4f}")
print("\n(Con pesos aleatorios, las predicciones son casi uniformes ≈ 0.25)")
print("Después del entrenamiento, las probabilidades del carácter correcto aumentarán.")

## Implementación con Código

### Funciones

In [ ]:
import numpy as np

def rnn_forward(inputs, h_prev, W_xh, W_hh, W_hy, b_h, b_y):
    """
    Forward pass de una RNN para una secuencia completa.
    inputs: lista de vectores one-hot
    h_prev: estado oculto inicial
    Retorna: outputs (probabilidades), hidden_states, cache
    """
    T = len(inputs)
    hidden_states = {}
    hidden_states[-1] = h_prev.copy()
    outputs = {}
    probs = {}

    for t in range(T):
        # Estado oculto
        hidden_states[t] = np.tanh(W_xh @ inputs[t] + W_hh @ hidden_states[t-1] + b_h)
        # Salida
        outputs[t] = W_hy @ hidden_states[t] + b_y
        # Softmax
        e_o = np.exp(outputs[t] - np.max(outputs[t]))
        probs[t] = e_o / e_o.sum()

    return probs, hidden_states, outputs


def rnn_backward(probs, hidden_states, inputs, targets,
                 W_xh, W_hh, W_hy):
    """
    Backward pass (BPTT) de una RNN.
    Retorna: gradientes de todos los pesos
    """
    T = len(inputs)
    d_W_xh = np.zeros_like(W_xh)
    d_W_hh = np.zeros_like(W_hh)
    d_W_hy = np.zeros_like(W_hy)
    d_b_h = np.zeros(W_hh.shape[0])
    d_b_y = np.zeros(W_hy.shape[0])
    d_h_next = np.zeros(W_hh.shape[0])

    loss = 0

    for t in reversed(range(T)):
        # Gradiente de softmax + cross-entropy
        d_o = probs[t].copy()
        d_o[targets[t]] -= 1  # ŷ - y
        loss += -np.log(probs[t][targets[t]] + 1e-10)

        # Gradiente de W_hy
        d_W_hy += np.outer(d_o, hidden_states[t])
        d_b_y += d_o

        # Gradiente hacia h_t
        d_h = W_hy.T @ d_o + d_h_next

        # Gradiente a través de tanh
        d_h_raw = (1 - hidden_states[t] ** 2) * d_h

        # Gradientes de W_xh y W_hh
        d_W_xh += np.outer(d_h_raw, inputs[t])
        d_W_hh += np.outer(d_h_raw, hidden_states[t-1])
        d_b_h += d_h_raw

        # Propagar hacia atrás en el tiempo
        d_h_next = W_hh.T @ d_h_raw

    return loss, d_W_xh, d_W_hh, d_W_hy, d_b_h, d_b_y


def clip_gradients(*grads, max_val=5):
    """Gradient clipping para evitar gradientes explosivos"""
    clipped = []
    for g in grads:
        clipped.append(np.clip(g, -max_val, max_val))
    return clipped


print("Funciones de RNN definidas: rnn_forward, rnn_backward, clip_gradients")

In [ ]:
import numpy as np

# Test rápido con las funciones
np.random.seed(42)

text = "hello"
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

hidden_size = 10

# Inicializar pesos
W_xh = np.random.randn(hidden_size, vocab_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
W_hy = np.random.randn(vocab_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)
b_y = np.zeros(vocab_size)

# Preparar secuencia
inputs = [np.eye(vocab_size)[char_to_idx[c]] for c in text[:-1]]  # "hell"
targets = [char_to_idx[c] for c in text[1:]]                       # "ello"
h_prev = np.zeros(hidden_size)

# Forward
probs, h_states, outputs = rnn_forward(inputs, h_prev, W_xh, W_hh, W_hy, b_h, b_y)

# Backward
loss, dW_xh, dW_hh, dW_hy, db_h, db_y = rnn_backward(
    probs, h_states, inputs, targets, W_xh, W_hh, W_hy
)

print(f"Texto: '{text}'")
print(f"Vocabulario: {chars} (tamaño: {vocab_size})")
print(f"Hidden size: {hidden_size}")
print(f"\nLoss inicial (antes de entrenar): {loss:.4f}")
print(f"Loss teórica con predicciones uniformes: {-np.log(1/vocab_size) * len(targets):.4f}")
print(f"\nNorma de gradientes:")
print(f"  |dW_xh| = {np.linalg.norm(dW_xh):.4f}")
print(f"  |dW_hh| = {np.linalg.norm(dW_hh):.4f}")
print(f"  |dW_hy| = {np.linalg.norm(dW_hy):.4f}")

### Clase

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class RNN:
    """
    RNN a nivel de carácter desde cero.
    Aprende a generar texto carácter por carácter.
    """
    def __init__(self, vocab_size, hidden_size=64, learning_rate=0.01,
                 seq_length=25, clip_val=5):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        self.seq_length = seq_length
        self.clip_val = clip_val
        self.loss_history = []

        # Inicializar pesos (Xavier)
        scale_xh = np.sqrt(2.0 / (vocab_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        scale_hy = np.sqrt(2.0 / (hidden_size + vocab_size))

        self.W_xh = np.random.randn(hidden_size, vocab_size) * scale_xh
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.W_hy = np.random.randn(vocab_size, hidden_size) * scale_hy
        self.b_h = np.zeros(hidden_size)
        self.b_y = np.zeros(vocab_size)

        # Para Adagrad
        self.mW_xh = np.zeros_like(self.W_xh)
        self.mW_hh = np.zeros_like(self.W_hh)
        self.mW_hy = np.zeros_like(self.W_hy)
        self.mb_h = np.zeros_like(self.b_h)
        self.mb_y = np.zeros_like(self.b_y)

    def _softmax(self, z):
        e_z = np.exp(z - np.max(z))
        return e_z / e_z.sum()

    def _forward(self, inputs, h_prev):
        """Forward pass para una secuencia"""
        T = len(inputs)
        xs, hs, os, ps = {}, {}, {}, {}
        hs[-1] = h_prev.copy()

        for t in range(T):
            xs[t] = np.zeros(self.vocab_size)
            xs[t][inputs[t]] = 1  # one-hot
            hs[t] = np.tanh(self.W_xh @ xs[t] + self.W_hh @ hs[t-1] + self.b_h)
            os[t] = self.W_hy @ hs[t] + self.b_y
            ps[t] = self._softmax(os[t])

        return xs, hs, ps

    def _backward(self, xs, hs, ps, targets):
        """Backward pass (BPTT)"""
        T = len(targets)
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h = np.zeros_like(self.b_h)
        db_y = np.zeros_like(self.b_y)
        dh_next = np.zeros(self.hidden_size)

        loss = 0
        for t in reversed(range(T)):
            loss += -np.log(ps[t][targets[t]] + 1e-10)

            # Gradiente softmax + CE
            do = ps[t].copy()
            do[targets[t]] -= 1

            dW_hy += np.outer(do, hs[t])
            db_y += do

            dh = self.W_hy.T @ do + dh_next
            dh_raw = (1 - hs[t] ** 2) * dh  # derivada de tanh

            dW_xh += np.outer(dh_raw, xs[t])
            dW_hh += np.outer(dh_raw, hs[t-1])
            db_h += dh_raw

            dh_next = self.W_hh.T @ dh_raw

        # Gradient clipping
        for d in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(d, -self.clip_val, self.clip_val, out=d)

        return loss, dW_xh, dW_hh, dW_hy, db_h, db_y

    def _update_weights(self, dW_xh, dW_hh, dW_hy, db_h, db_y):
        """Actualizar pesos con Adagrad"""
        for param, dparam, mem in zip(
            [self.W_xh, self.W_hh, self.W_hy, self.b_h, self.b_y],
            [dW_xh, dW_hh, dW_hy, db_h, db_y],
            [self.mW_xh, self.mW_hh, self.mW_hy, self.mb_h, self.mb_y]
        ):
            mem += dparam * dparam
            param -= self.lr * dparam / (np.sqrt(mem) + 1e-8)

    def fit(self, text, char_to_idx, idx_to_char, epochs=1, verbose=True):
        """
        Entrena la RNN en un texto.
        text: string de entrenamiento
        char_to_idx: diccionario carácter → índice
        idx_to_char: diccionario índice → carácter
        """
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        data = [char_to_idx[c] for c in text]
        n = len(data)

        for epoch in range(epochs):
            h_prev = np.zeros(self.hidden_size)
            epoch_loss = 0
            n_batches = 0
            p = 0  # posición en el texto

            while p + self.seq_length + 1 < n:
                # Extraer secuencia
                inputs = data[p:p+self.seq_length]
                targets = data[p+1:p+self.seq_length+1]

                # Forward
                xs, hs, ps = self._forward(inputs, h_prev)

                # Backward
                loss, dW_xh, dW_hh, dW_hy, db_h, db_y = self._backward(
                    xs, hs, ps, targets
                )

                # Update
                self._update_weights(dW_xh, dW_hh, dW_hy, db_h, db_y)

                epoch_loss += loss
                n_batches += 1
                h_prev = hs[self.seq_length - 1]
                p += self.seq_length

            avg_loss = epoch_loss / max(n_batches, 1)
            self.loss_history.append(avg_loss)

            if verbose and (epoch + 1) % max(1, epochs // 10) == 0:
                sample = self.generate(data[0], length=50)
                print(f"Época {epoch+1:4d}/{epochs} — Loss: {avg_loss:.4f} — Muestra: {sample[:60]}")

    def generate(self, seed_idx, length=100, temperature=1.0):
        """
        Genera texto carácter por carácter.
        seed_idx: índice del carácter semilla
        length: número de caracteres a generar
        temperature: >1 más aleatorio, <1 más determinístico
        """
        h = np.zeros(self.hidden_size)
        x_idx = seed_idx
        result = []

        for _ in range(length):
            x = np.zeros(self.vocab_size)
            x[x_idx] = 1

            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            o = self.W_hy @ h + self.b_y

            # Aplicar temperatura
            o = o / temperature
            p = self._softmax(o)

            # Muestrear
            x_idx = np.random.choice(self.vocab_size, p=p)
            result.append(self.idx_to_char[x_idx])

        return ''.join(result)

    def score(self, text):
        """Calcula la pérdida promedio en un texto"""
        data = [self.char_to_idx[c] for c in text]
        h = np.zeros(self.hidden_size)
        total_loss = 0

        for t in range(len(data) - 1):
            x = np.zeros(self.vocab_size)
            x[data[t]] = 1
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            o = self.W_hy @ h + self.b_y
            p = self._softmax(o)
            total_loss += -np.log(p[data[t+1]] + 1e-10)

        return total_loss / (len(data) - 1)

    def plot_loss(self):
        """Grafica la curva de pérdida"""
        plt.figure(figsize=(10, 4))
        plt.plot(range(1, len(self.loss_history)+1), self.loss_history,
                 color='#e74c3c', linewidth=1.5, alpha=0.8)
        plt.xlabel('Época', fontsize=12)
        plt.ylabel('Loss promedio (Cross-Entropy)', fontsize=12)
        plt.title('Curva de Entrenamiento de la RNN', fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
import numpy as np

# Datos de entrenamiento: texto en español
text = """la inteligencia artificial es una rama de la informatica que busca crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana como el aprendizaje la percepcion el razonamiento y la toma de decisiones las redes neuronales son modelos inspirados en el cerebro humano que aprenden patrones a partir de datos el aprendizaje profundo utiliza redes con muchas capas para resolver problemas complejos como el reconocimiento de imagenes y el procesamiento del lenguaje natural"""

# Crear vocabulario
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

print(f"Texto de entrenamiento: {len(text)} caracteres")
print(f"Vocabulario: {vocab_size} caracteres únicos")
print(f"Caracteres: {''.join(chars)}")

In [ ]:
# Entrenar la RNN
np.random.seed(42)
model = RNN(
    vocab_size=vocab_size,
    hidden_size=64,
    learning_rate=0.01,
    seq_length=25,
    clip_val=5
)

model.fit(text, char_to_idx, idx_to_char, epochs=200, verbose=True)

In [ ]:
# Visualizar curva de pérdida
model.plot_loss()

In [ ]:
# Generar texto con diferentes temperaturas
print("="*60)
print("GENERACIÓN DE TEXTO CON DIFERENTES TEMPERATURAS")
print("="*60)

seed = char_to_idx['l']

for temp in [0.3, 0.7, 1.0, 1.5]:
    generated = model.generate(seed, length=100, temperature=temp)
    print(f"\nTemperatura = {temp}:")
    print(f"  l{generated}")

print("\n" + "-"*60)
print("Temp baja (0.3) → texto más repetitivo pero coherente")
print("Temp alta (1.5) → texto más diverso pero menos coherente")

In [ ]:
# Visualizar cómo evoluciona el estado oculto durante la lectura de texto
sample_text = "inteligencia artificial"

h = np.zeros(model.hidden_size)
hidden_states = [h.copy()]

for c in sample_text:
    x = np.zeros(model.vocab_size)
    x[char_to_idx[c]] = 1
    h = np.tanh(model.W_xh @ x + model.W_hh @ h + model.b_h)
    hidden_states.append(h.copy())

hidden_matrix = np.array(hidden_states).T  # (hidden_size, T+1)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(hidden_matrix[:20, :], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Paso temporal (carácter)', fontsize=12)
ax.set_ylabel('Neurona oculta', fontsize=12)
ax.set_xticks(range(len(sample_text) + 1))
ax.set_xticklabels(['h₀'] + list(sample_text), fontsize=9)
ax.set_title(f'Evolución del Estado Oculto al Leer: "{sample_text}"',
             fontsize=13, fontweight='bold')
plt.colorbar(im, label='Activación (tanh)', shrink=0.8)
plt.tight_layout()
plt.show()

print("Cada columna muestra el 'estado mental' de la RNN después de leer cada carácter.")
print("Nótese cómo el patrón cambia al pasar de una palabra a otra (espacio).")

In [ ]:
# Visualizar las probabilidades de predicción en cada paso
sample = "la red"

h = np.zeros(model.hidden_size)
predictions = []

for c in sample:
    x = np.zeros(model.vocab_size)
    x[char_to_idx[c]] = 1
    h = np.tanh(model.W_xh @ x + model.W_hh @ h + model.b_h)
    o = model.W_hy @ h + model.b_y
    p = model._softmax(o)
    predictions.append(p)

# Mostrar top-5 predicciones para cada paso
print(f"Predicciones al leer '{sample}':")
print("="*60)
for t, (c, p) in enumerate(zip(sample, predictions)):
    top_5 = np.argsort(p)[::-1][:5]
    print(f"\nDespués de '{sample[:t+1]}', siguiente carácter más probable:")
    for idx in top_5:
        char_repr = repr(idx_to_char[idx])
        bar = '█' * int(p[idx] * 40)
        print(f"  {char_repr:5s} {p[idx]:.3f} {bar}")

In [ ]:
# Comparar rendimiento con diferentes tamaños de estado oculto
print("="*60)
print("COMPARACIÓN: Diferentes tamaños de estado oculto")
print("="*60)

hidden_sizes = [16, 32, 64]
results = []

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, hs in zip(axes, hidden_sizes):
    np.random.seed(42)
    m = RNN(vocab_size=vocab_size, hidden_size=hs, learning_rate=0.01,
            seq_length=25, clip_val=5)
    m.fit(text, char_to_idx, idx_to_char, epochs=100, verbose=False)

    final_loss = m.loss_history[-1]
    sample = m.generate(char_to_idx['l'], length=80, temperature=0.7)

    n_params = hs * vocab_size + hs * hs + vocab_size * hs + hs + vocab_size
    results.append((hs, final_loss, n_params, sample))

    ax.plot(m.loss_history, color='#e74c3c', linewidth=1.5)
    ax.set_title(f'hidden={hs}\nLoss final: {final_loss:.2f}\nParams: {n_params:,}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

plt.suptitle('Efecto del Tamaño del Estado Oculto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTexto generado por cada modelo (temp=0.7):")
print("-"*60)
for hs, loss, n_params, sample in results:
    print(f"\nhidden={hs} (params={n_params:,}, loss={loss:.2f}):")
    print(f"  l{sample}")

In [ ]:
# Resumen
print("="*70)
print("RESUMEN: REDES NEURONALES RECURRENTES (RNN)")
print("="*70)
print()
print("┌─────────────────────────────────────────────────────────────────────┐")
print("│ RNN                                                                 │")
print("│   • Procesa secuencias manteniendo un estado oculto (memoria)       │")
print("│   • Ecuación: h_t = tanh(W_xh·x_t + W_hh·h_{t-1} + b)             │")
print("│   • Los pesos se comparten a lo largo del tiempo                    │")
print("│   • Se entrena con BPTT (Backpropagation Through Time)              │")
print("├─────────────────────────────────────────────────────────────────────┤")
print("│ Aplicaciones                                                        │")
print("│   • Modelado de lenguaje (predecir siguiente palabra/carácter)      │")
print("│   • Traducción automática (seq2seq)                                 │")
print("│   • Series temporales y predicción                                  │")
print("│   • Reconocimiento de voz                                           │")
print("├─────────────────────────────────────────────────────────────────────┤")
print("│ Limitaciones → Soluciones                                           │")
print("│   • Gradiente desvaneciente → LSTM (Long Short-Term Memory)         │")
print("│   • Gradiente explosivo → Gradient clipping                         │")
print("│   • Procesamiento secuencial (lento) → Transformer (paralelo)       │")
print("│   • Memoria limitada → Attention mechanism                          │")
print("├─────────────────────────────────────────────────────────────────────┤")
print("│ Evolución histórica                                                 │")
print("│   RNN → LSTM/GRU → Seq2Seq → Attention → Transformer → GPT/BERT   │")
print("└─────────────────────────────────────────────────────────────────────┘")